<div align="center">

# VOID: Video Object and Interaction Deletion

**Interaction-aware video object removal using CogVideoX**

[Project Page](https://void-model.github.io/) | [Paper](https://arxiv.org/abs/XXXX.XXXXX) | [HuggingFace](https://huggingface.co/netflix/void-model)

</div>

This notebook runs **VOID Pass 1** inference on a sample video. VOID removes objects from videos along with all interactions they induce — not just shadows and reflections, but physical interactions like objects falling when a support is removed.

**Requirements:** A GPU with **40GB+ VRAM** (e.g., A100). CPU offloading + FP8 quantization is used by default to reduce memory.

## 1. Setup

In [6]:
# Clone the repo (skip if already cloned)
!git clone https://github.com/netflix/void-model.git 2>/dev/null || echo "Repo already exists"
%cd void-model

/content/void-model/void-model


In [7]:
!pip install -q -r requirements.txt

In [8]:
# Download the base CogVideoX inpainting model
!hf download alibaba-pai/CogVideoX-Fun-V1.5-5b-InP --local-dir ./CogVideoX-Fun-V1.5-5b-InP --quiet

/content/void-model/void-model/CogVideoX-Fun-V1.5-5b-InP


In [9]:
# Download the VOID Pass 1 checkpoint
!hf download netflix/void-model void_pass1.safetensors --local-dir . --quiet

void_pass1.safetensors


## 2. Load the Model

In [12]:
import os
import json
import torch
import numpy as np
import torch.nn.functional as F
from safetensors.torch import load_file
from diffusers import DDIMScheduler
from IPython.display import Video, display

from videox_fun.models import (
    AutoencoderKLCogVideoX,
    CogVideoXTransformer3DModel,
    T5EncoderModel,
    T5Tokenizer,
)
from videox_fun.pipeline import CogVideoXFunInpaintPipeline
from videox_fun.utils.fp8_optimization import convert_weight_dtype_wrapper
from videox_fun.utils.utils import get_video_mask_input, save_videos_grid, save_inout_row

RuntimeError: Failed to import diffusers.schedulers.scheduling_ddim because of the following error (look up to see its traceback):
numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# --- Configuration ---
BASE_MODEL_PATH = "./CogVideoX-Fun-V1.5-5b-InP"
TRANSFORMER_CKPT = "./void_pass1.safetensors"
SAMPLE_NAME = "lime"           # Try "moving_ball" or "pillow" too
DATA_ROOTDIR = "./sample"      # Directory containing sample folders

SAMPLE_SIZE = (384, 672)       # (H, W)
MAX_VIDEO_LENGTH = 197
TEMPORAL_WINDOW_SIZE = 85
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 1.0
SEED = 42
DEVICE = "cuda"
WEIGHT_DTYPE = torch.bfloat16

In [ ]:
# Load VAE
vae = AutoencoderKLCogVideoX.from_pretrained(
    BASE_MODEL_PATH, subfolder="vae"
).to(WEIGHT_DTYPE)

# Compute effective video length (must align with VAE temporal compression)
video_length = int((MAX_VIDEO_LENGTH - 1) // vae.config.temporal_compression_ratio * vae.config.temporal_compression_ratio) + 1
print(f"Effective video length: {video_length}")

# Load transformer from base model with VAE mask channels, then load VOID weights
transformer = CogVideoXTransformer3DModel.from_pretrained(
    BASE_MODEL_PATH, subfolder="transformer",
    low_cpu_mem_usage=True, use_vae_mask=True,
).to(WEIGHT_DTYPE)

print(f"Loading VOID checkpoint from {TRANSFORMER_CKPT}...")
state_dict = load_file(TRANSFORMER_CKPT)

# Handle potential channel dimension mismatch for VAE mask
param_name = "patch_embed.proj.weight"
if state_dict[param_name].size(1) != transformer.state_dict()[param_name].size(1):
    latent_ch, feat_scale = 16, 8
    feat_dim = latent_ch * feat_scale
    new_weight = transformer.state_dict()[param_name].clone()
    new_weight[:, :feat_dim] = state_dict[param_name][:, :feat_dim]
    new_weight[:, -feat_dim:] = state_dict[param_name][:, -feat_dim:]
    state_dict[param_name] = new_weight
    print(f"Adapted {param_name} channels for VAE mask")

m, u = transformer.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {len(m)}, Unexpected keys: {len(u)}")

# Load tokenizer & text encoder
tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL_PATH, subfolder="tokenizer")
text_encoder = T5EncoderModel.from_pretrained(
    BASE_MODEL_PATH, subfolder="text_encoder", torch_dtype=WEIGHT_DTYPE
)

# Load scheduler (DDIM, matching the default config)
scheduler = DDIMScheduler.from_pretrained(BASE_MODEL_PATH, subfolder="scheduler")

print("All components loaded.")

In [ ]:
# Assemble pipeline with CPU offloading + FP8 quantization
pipe = CogVideoXFunInpaintPipeline(
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    vae=vae,
    transformer=transformer,
    scheduler=scheduler,
)

convert_weight_dtype_wrapper(pipe.transformer, WEIGHT_DTYPE)
pipe.enable_model_cpu_offload(device=DEVICE)

generator = torch.Generator(device=DEVICE).manual_seed(SEED)
print("Pipeline ready.")

## 3. Prepare Input Data

Each sample folder contains:
- `input_video.mp4` — source video
- `quadmask_0.mp4` — 4-value semantic mask (0=remove, 63=overlap, 127=affected, 255=keep)
- `prompt.json` — background description after removal

In [ ]:
# Load input video, quadmask, and prompt using the same utility as the inference script
input_video, input_video_mask, prompt, _ = get_video_mask_input(
    SAMPLE_NAME,
    sample_size=SAMPLE_SIZE,
    keep_fg_ids=[-1],
    max_video_length=video_length,
    temporal_window_size=TEMPORAL_WINDOW_SIZE,
    data_rootdir=DATA_ROOTDIR,
    use_quadmask=True,
    dilate_width=11,
)

negative_prompt = (
    "The video is not of a high quality, it has a low resolution. "
    "Watermark present in each frame. The background is solid. "
    "Strange body and strange trajectory. Distortion. "
)

print(f"Prompt: {prompt}")
print(f"Video: {input_video.shape}, Mask: {input_video_mask.shape}")

In [ ]:
# Preview input video
input_video_path = os.path.join(DATA_ROOTDIR, SAMPLE_NAME, "input_video.mp4")
display(Video(input_video_path, embed=True, width=672))

## 4. Run Inference (Pass 1)

In [ ]:
with torch.no_grad():
    sample = pipe(
        prompt,
        num_frames=TEMPORAL_WINDOW_SIZE,
        negative_prompt=negative_prompt,
        height=SAMPLE_SIZE[0],
        width=SAMPLE_SIZE[1],
        generator=generator,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        video=input_video,
        mask_video=input_video_mask,
        strength=1.0,
        use_trimask=True,
        use_vae_mask=True,
    ).videos

print(f"Output shape: {sample.shape}")

## 5. Save & Display Result

In [ ]:
output_path = "./output_void.mp4"
save_videos_grid(sample, output_path, fps=12)
print(f"Saved to {output_path}")

display(Video(output_path, embed=True, width=672))

In [ ]:
# Side-by-side comparison (input | mask | output)
comparison_path = "./output_comparison.mp4"
save_inout_row(input_video, input_video_mask, sample, comparison_path, fps=12)
print(f"Comparison saved to {comparison_path}")

display(Video(comparison_path, embed=True, width=1344))

## Notes

- **Pass 2 (optional):** For better temporal consistency, run the Pass 2 warped-noise refinement. See the [README](https://github.com/netflix/void-model#-pipeline) for details.
- **Custom videos:** To use your own videos, prepare the input format (video + quadmask + prompt) using the `VLM-MASK-REASONER/` pipeline included in the repo.
- **Memory:** The default mode (`model_cpu_offload` + FP8) requires ~40GB VRAM. For lower VRAM, try `pipe.enable_sequential_cpu_offload()` instead (slower but uses less GPU memory).